## HTTP Post

### Concept first client will sent the http request in json format called response body . then we will validate that data using pydantic and raise any error if present and if not register into new record

In [11]:
from fastapi import FastAPI,HTTPException
app = FastAPI()

In [5]:
from pydantic import BaseModel,Field,computed_field
from typing import Annotated,Literal

In [6]:
class Patient(BaseModel):
    id : Annotated[str,Field(...,description='id of the patient',examples=['P001'])]
    name : Annotated[str,Field(...,description='Name of the patient')]
    city : Annotated[str,Field(...,description='city of the patient')]
    age : Annotated[int,Field(...,description='age of the patient',gt=0,lt=120)]
    gender : Annotated[Literal['male','female','other'],Field(...,description='the gender of the patient')]
    height : Annotated[float,Field(...,description='the height of the patient in m' , gt = 0)]
    weight : Annotated[float,Field(...,description='the weight of the patient in kg ',gt = 0)]

    @computed_field
    @property
    def bmi(self)->float:
        return round(self.weight/self.height**2,2)

    @computed_field
    @property
    def verdict(self)->str:
        if self.bmi < 18.5:
            return 'Underweight'
        elif self.bmi < 30:
            return 'Normal'
        else:
            return 'Obese'


In [7]:
import json

In [8]:
def load_data():
    with open('patients.json','r') as f:
        data = json.load(f)
    return data

In [10]:
def save_date(data):
    with open('patients.json','w') as f:
        json.dump(data,f)

In [12]:
from fastapi.responses import JSONResponse

In [13]:
@app.post('/create')
def create_patient(patient: Patient):
    # load the patient data 
    data = load_data()

    #check if patient already exists
    if patient.id in data:
        raise HTTPException(status_code=400,detail='patient already exists.')

    # add new patient in the database
    data[patient.id] = patient.model_dump(exclude=['id'])

    # save into the json file
    save_date(data)

    return JSONResponse(status_code=201,content='patient created successfully')
